# MPC against PID

Works through the comparison from the raw results, so the claims in the README
can be checked rather than taken on trust. Everything here reads
`results/runs.csv`, which `analysis/run_comparison.py` writes.

The questions, in order:

1. Which controller wins, and by how much?
2. Is that a real difference or noise between seeds?
3. Does it survive on scenarios the gains were **not** tuned on?
4. *Where* does the difference come from — and where does it disappear?
5. What does it cost?

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(ROOT, "src"))

plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
PID, MPC = "#4C6EF5", "#E8590C"

runs = pd.read_csv(os.path.join(ROOT, "results", "runs.csv"))
print(f"{len(runs)} runs, {len(runs) // 2} paired scenarios")
runs.head()

## 1. Pair the runs

Pairing is what makes this sensitive. Averaging each controller across the grid
and subtracting would bury a consistent difference under the variance *between
scenarios*, which is far larger than the variance *between controllers on the
same scenario*.

Every pair shares a seed, so the two controllers met the same gusts and the
same sensor noise, sample for sample.

In [ ]:
keys = ["mission", "speed", "wind", "corner_mode", "seed"]
wide = runs.pivot_table(index=keys, columns="tracker",
                        values=["rms_error_m", "max_error_m", "rms_cross_track_m",
                                "rms_along_track_m", "rms_jerk_ms3",
                                "mean_solve_time_us", "saturated_fraction",
                                "held_out", "completed"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
wide["ratio"] = wide["rms_error_m_pid"] / wide["rms_error_m_mpc"]
wide["winner"] = np.where(wide["ratio"] > 1.0, "mpc", "pid")

print(f"MPC wins {int((wide['winner'] == 'mpc').sum())} of {len(wide)} pairs")
print(f"median error ratio (PID / MPC): {wide['ratio'].median():.3f}")
wide[keys + ["rms_error_m_pid", "rms_error_m_mpc", "ratio", "winner"]].head(12)

## 2. Is it real, or is it seed noise?

A paired comparison lets us ask this properly. If the two controllers were
equivalent, the sign of the per-pair difference would be a coin flip. Rather
than lean on a t-test and its normality assumption, this uses the sign test,
which assumes almost nothing.

In [ ]:
from scipy import stats

difference = wide["rms_error_m_pid"] - wide["rms_error_m_mpc"]
wins = int((difference > 0).sum())
n = int((difference != 0).sum())
p_sign = stats.binomtest(wins, n, 0.5).pvalue
p_wilcoxon = stats.wilcoxon(wide["rms_error_m_pid"], wide["rms_error_m_mpc"]).pvalue

print(f"MPC better in {wins}/{n} pairs")
print(f"sign test          p = {p_sign:.2e}")
print(f"Wilcoxon signed-rank p = {p_wilcoxon:.2e}")
print()
print("A small p here says the *direction* is consistent. It says nothing about")
print("whether the size of the difference matters operationally, which is a")
print("judgement about the job, not a statistic.")

fig, ax = plt.subplots()
order = np.argsort(wide["ratio"].values)
colours = [MPC if r > 1 else PID for r in wide["ratio"].values[order]]
ax.bar(range(len(order)), wide["ratio"].values[order], color=colours)
ax.axhline(1.0, color="#212529", linewidth=1.0)
ax.set_xlabel("paired scenario (sorted)")
ax.set_ylabel("PID error / MPC error")
ax.set_title("above 1 means the MPC tracked closer")
plt.show()

## 3. Tuned against held out

Gains were tuned on `facade_inspection`, nominal speed, calm air, and nowhere
else. If the advantage only appeared there, it would be a tuning artefact.

In [ ]:
for label, subset in (("tuned on", wide[wide["held_out_pid"] == 0]),
                      ("held out", wide[wide["held_out_pid"] == 1])):
    if not len(subset):
        continue
    print(f"{label:>9}: {len(subset):2d} pairs, "
          f"MPC wins {int((subset['winner'] == 'mpc').sum()):2d}, "
          f"median ratio {subset['ratio'].median():.3f}, "
          f"PID {subset['rms_error_m_pid'].mean():.4f} m, "
          f"MPC {subset['rms_error_m_mpc'].mean():.4f} m")

## 4. Where does the difference come from?

This is the part worth understanding. The MPC's structural advantage is
**preview**: it samples the reference across its whole horizon, so it starts
leaning into a corner before the corner has produced any error at all. A PID is
causal in the reference — at the moment it would need to start leaning, its
error is still zero.

That predicts the gap should widen where the reference changes direction
sharply and at speed, and narrow on long straight legs where there is nothing
to preview.

In [ ]:
for column in ("mission", "speed", "wind", "corner_mode"):
    print(f"--- by {column}")
    grouped = wide.groupby(column).agg(
        pairs=("ratio", "size"),
        mpc_wins=("winner", lambda s: int((s == "mpc").sum())),
        median_ratio=("ratio", "median"),
        pid_rms=("rms_error_m_pid", "mean"),
        mpc_rms=("rms_error_m_mpc", "mean"),
    )
    print(grouped.to_string(float_format=lambda v: f"{v:.4f}"))
    print()

In [ ]:
# Cross-track against along-track. The MPC's preview should show up mostly in
# cross-track error, which is the component that matters near a structure.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric, label in (
        (axes[0], "rms_cross_track_m", "cross-track RMS (m)"),
        (axes[1], "rms_along_track_m", "along-track RMS (m)")):
    ax.scatter(wide[f"{metric}_pid"], wide[f"{metric}_mpc"], s=26, alpha=0.75,
               color="#5F3DC4")
    limit = max(wide[f"{metric}_pid"].max(), wide[f"{metric}_mpc"].max()) * 1.05
    ax.plot([0, limit], [0, limit], "k--", linewidth=0.9)
    ax.set_xlabel(f"PID {label}"); ax.set_ylabel(f"MPC {label}")
    ax.set_title("points below the line favour the MPC")
plt.tight_layout(); plt.show()

### The infeasible reference

`sharp` corner mode produces a reference with a velocity step no vehicle can
follow. It is in the grid deliberately: it is the case where preview cannot
help, because there is nothing achievable to preview, and it tests what each
controller does when asked for the impossible. A controller that chases it
harder is not better.

In [ ]:
sharp = wide[wide["corner_mode"] == "sharp"]
if len(sharp):
    print(sharp[["mission", "wind", "seed", "rms_error_m_pid", "rms_error_m_mpc",
                 "ratio", "saturated_fraction_pid", "saturated_fraction_mpc"]]
          .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print()
    print("Watch the saturated fraction as much as the error: a controller that is")
    print("saturated more of the time is spending more of the flight at the vehicle's")
    print("limits, which is where nothing has any margin left.")
else:
    print("no sharp-corner runs (was the comparison run with --quick?)")

## 5. What it costs

Two costs, and both matter: compute per control step, and how rough the demand
is. A controller that buys accuracy with a sawtooth acceleration demand is
paying for it in actuator wear and battery.

In [ ]:
print("compute per control step (the 50 Hz period is 20 000 us)")
print(runs.groupby("tracker")[["mean_solve_time_us", "mean_update_time_us",
                               "p99_update_time_us"]]
      .mean().to_string(float_format=lambda v: f"{v:9.1f}"))
print()
print("smoothness of the acceleration demand")
print(runs.groupby("tracker")[["rms_accel_demand_ms2", "rms_jerk_ms3",
                               "saturated_fraction"]]
      .mean().to_string(float_format=lambda v: f"{v:9.3f}"))

fig, ax = plt.subplots()
for tracker, colour in (("pid", PID), ("mpc", MPC)):
    subset = runs[runs["tracker"] == tracker]
    ax.scatter(subset["rms_jerk_ms3"], subset["rms_error_m"], s=24, alpha=0.75,
               color=colour, label=tracker.upper())
ax.set_xlabel("RMS jerk of the acceleration demand (m/s^3)")
ax.set_ylabel("RMS position error (m)")
ax.set_title("accuracy against smoothness; down and left is better")
ax.legend(frameon=False)
plt.show()

## The estimator is shared

Worth confirming rather than assuming: both controllers fly on the same
estimator, so any difference between them is not an estimation difference. If
these two rows differ materially, something in the harness is not paired.

In [ ]:
print(runs.groupby("tracker")[["estimator_rms_m", "estimator_mean_nis",
                               "fraction_rtk_fixed"]]
      .mean().to_string(float_format=lambda v: f"{v:9.4f}"))
print()
print("They are not identical, and should not be: the two controllers fly slightly")
print("different trajectories, so the estimator sees slightly different motion. What")
print("matters is that neither is systematically better served by it.")

## Reading the result honestly

Things this comparison does **not** establish:

- That MPC is better in general. It is better *on these missions, on this
  airframe, with this reference generator*, and the mechanism is preview.
- That the absolute errors would hold on a real aircraft. The plant is a
  simplification; see `docs/LIMITATIONS.md`.
- That either controller is optimally tuned. Both were tuned by the same
  procedure with the same budget, which makes the comparison fair without making
  either one best.

And the cost side is real: the MPC is roughly an order of magnitude more compute
per step, needs a QP solver on the vehicle, and has more parameters to get
wrong. On a long straight survey where the gap nearly closes, that is a poor
trade.